# **Computer Vision based bikeability Score**

## 1. Setup & Imports

In [1]:
import csv
import datetime as dt
import math
from pathlib import Path
import cv2
import numpy as np

## 2. Konfiguration: Bewertungstabellen & Gewichte

Der Score setzt sich aus **drei Teilkomponenten** zusammen, die jeweils von einem eigenen Modell
kommen:

- **Ground Detection** → Oberflächenklasse (Asphalt, Schotter, Erde, …)
- **Environment Detection** → Umgebungsklasse (Natur, Stadt, Industrie, …)
- **Object Detection** → Objektzählungen (Autos, Fußgänger, Radfahrer, …)

Jede Klasse bekommt eine feste Bewertung in `[0, 1]`. Objekte erhalten einen *Störfaktor*
(negativ = fahrradfreundlich, z. B. andere Radfahrer). Die drei Subscores werden über die
Gewichte `W_GROUND + W_ENV + W_OBJECT = 1` zum Gesamtscore kombiniert – dadurch bleibt der Score
**garantiert** in `[0, 100]`.

> Die Werte sind heuristisch


In [24]:
# --- Gewichte der drei Teilkomponenten (Summe = 1.0) ---
W_GROUND = 0.2
W_ENV = 0.45
W_OBJECT = 0.35

GROUND_QUALITY: dict[str, float] = {
    "asphalt": 1.0,
    "cobblestone": 0.7,
    "gravel": 0.4,
    "dirt": 0.1,
}

ENV_QUALITY: dict[str, float] = {
    "vegetation": 1.0,
    "water": 1.0,
    "city": 0.4,
}

OBJECT_PENALTY: dict[str, float] = {
    "car": 1.0,
    "bus": 2.0,
    "person": 0.3,
    "bicycle": -0.5,
}

print("Konfiguration geladen.")
print(f"  Gewichte: ground={W_GROUND}, env={W_ENV}, object={W_OBJECT} "
      f"(Summe={W_GROUND + W_ENV + W_OBJECT:.2f})")


Konfiguration geladen.
  Gewichte: ground=0.2, env=0.45, object=0.35 (Summe=1.00)


## 3. Bikeability-Score-Funktion

Aus den drei Modell-Ausgaben (Oberfläche, Umgebung, Objektzählungen) wird pro Frame ein Score
in `[0, 100]` berechnet:

$$
A = 100 \cdot \Big( w_g \cdot S_{\text{ground}} + w_e \cdot S_{\text{env}} + w_o \cdot S_{\text{object}} \Big)
$$

- **Ground/Env-Subscore:** direkter Lookup der Klassenqualität (bei Anteils-Verteilungen der
  gewichtete Mittelwert).
- **Object-Subscore:** Sättigungsfunktion $e^{-\sum_k \lambda_k n_k}$ – robust gegen Ausreißer,
  ein einzelnes überfülltes Frame dominiert dadurch nicht.


In [ ]:
def _shares_from(value) -> dict[str, float]:
    if isinstance(value, str):
        return {value: 1.0}
    if isinstance(value, dict):
        total = sum(value.values()) or 1.0
        return {k: v / total for k, v in value.items()}
    labels = list(value)
    total = len(labels) or 1
    counts: dict[str, float] = {}
    for label in labels:
        counts[label] = counts.get(label, 0.0) + 1.0
    return {k: v / total for k, v in counts.items()}


def ground_subscore(ground_result) -> float:
    """Oberflächen-Subscore in [0, 1] aus der Ground-Detection-Ausgabe."""
    shares = _shares_from(ground_result)
    return sum(GROUND_QUALITY.get(k, 0.5) * v for k, v in shares.items())


def env_subscore(env_result) -> float:
    """Umgebungs-Subscore in [0, 1] aus der Environment-Detection-Ausgabe."""
    shares = _shares_from(env_result)
    return sum(ENV_QUALITY.get(k, 0.5) * v for k, v in shares.items())


def object_subscore(object_counts: dict[str, int], n_frames: int = 1) -> float:
    """Objekt-Subscore in (0, 1] aus den Objektzählungen (Sättigungsfunktion)."""
    density = sum(
        OBJECT_PENALTY.get(k, 0.0) * n for k, n in object_counts.items()
    ) / max(n_frames, 1)
    return math.exp(-max(density, 0.0))


def bikeability_score(
    ground_result,
    env_result,
    object_counts: dict[str, int],
    n_frames: int = 1,
) -> float:
    s_g = ground_subscore(ground_result)
    s_e = env_subscore(env_result)
    s_o = object_subscore(object_counts, n_frames)
    score = 100.0 * (W_GROUND * s_g + W_ENV * s_e + W_OBJECT * s_o)
    return round(score, 2)


# --- Kurzer Funktionstest mit Beispielwerten ---
_demo = bikeability_score(
    ground_result="asphalt",
    env_result="vegetation",
    object_counts={"car": 2, "bicycle": 2},
)
print(f"Demo-Score (Asphalt, Vegetation, 2 Autos, 2 Räder): {_demo}/100")


Demo-Score (Asphalt, Vegetation, 1 Auto, 0 Räder): 77.88/100


## 4. Video-Verarbeitung: Score alle 5 Sekunden

Die Funktion `process_video`:

1. öffnet das Video und ermittelt die **FPS**,
2. extrahiert **alle 5 Sekunden** genau ein Frame,
3. ruft für jedes Frame die drei Modelle auf *(Aufrufe zunächst auskommelt – Platzhalter
   liefern Dummy-Werte, bis die Modelle angebunden sind)*,
4. berechnet damit den **Bikeability-Score**,
5. speichert jede Zeile mit **Timestamp** in eine **CSV** (für späteres Mergen mit GPX) und
   sammelt die Ergebnisse zusätzlich in einem **Array**.

Der Timestamp wird sowohl in **Sekunden ab Videostart** als auch als `HH:MM:SS` gespeichert. Ist
eine reale Startzeit des Videos bekannt, wird zusätzlich ein absoluter ISO-Zeitstempel
geschrieben – ideal zum Zusammenführen mit GPX-Track-Punkten.


In [ ]:
CSV_FIELDS = [
    "frame_index",
    "time_seconds",
    "timestamp",
    "abs_timestamp",
    "ground",
    "environment",
    "objects",
    "score",
]


def process_video(
    video_path: str | Path,
    output_csv: str | Path,
    interval_seconds: float = 5.0,
    video_start_time: dt.datetime | None = None,
) -> list[dict]:

    video_path = Path(video_path)
    output_csv = Path(output_csv)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Video konnte nicht geöffnet werden: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30.0
        print("⚠ FPS konnten nicht gelesen werden - Fallback auf 30 fps.")

    frame_interval = max(1, round(fps * interval_seconds))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video      : {video_path.name}")
    print(f"FPS        : {fps:.2f}")
    print(f"Frames ges.: {total_frames}")
    print(f"Auswertung : alle {interval_seconds:.0f}s  ->  jedes {frame_interval}. Frame")
    print("-" * 50)

    results: list[dict] = []
    frame_index = 0

    with output_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % frame_interval == 0:
                time_seconds = frame_index / fps

                # --- Modelle aufrufen (noch auskommentiert) ---------------------------
                # ground_result = detect_ground(frame)          # -> "asphalt" / [...] / {...}
                # env_result = detect_environment(frame)        # -> "city" / [...] / {...}
                # object_counts = detect_objects(frame)         # -> {"car": 5, "bicycle": 0}

                # --- Platzhalter, bis die Modelle angebunden sind ---------------------
                ground_result = "unknown"
                env_result = "unknown"
                object_counts: dict[str, int] = {}
                # ---------------------------------------------------------------------

                score = bikeability_score(ground_result, env_result, object_counts)

                abs_timestamp = ""
                if video_start_time is not None:
                    abs_timestamp = (
                        video_start_time + dt.timedelta(seconds=time_seconds)
                    ).isoformat()

                row = {
                    "frame_index": frame_index,
                    "time_seconds": round(time_seconds, 3),
                    "timestamp": str(dt.timedelta(seconds=int(time_seconds))),
                    "abs_timestamp": abs_timestamp,
                    "ground": ground_result if isinstance(ground_result, str) else "",
                    "environment": env_result if isinstance(env_result, str) else "",
                    "objects": ";".join(f"{k}={v}" for k, v in object_counts.items()),
                    "score": score,
                }
                writer.writerow(row)
                results.append(row)

            frame_index += 1

    cap.release()
    print(f"Fertig: {len(results)} Frames ausgewertet -> {output_csv.resolve()}")
    return results


## 5. Ausführung & Durchschnitts-Score

Video verarbeiten, Ergebnisse als Array + CSV erhalten und den **durchschnittlichen
Bikeability-Score** über alle ausgewerteten Frames berechnen.


In [ ]:
VIDEO_PATH = Path("dataset/raw/DJI_0375.MP4")
OUTPUT_CSV = Path("dataset/bikeability_scores.csv")

VIDEO_START_TIME = None

# --- Verarbeitung starten -------------------------------------------------------
scores = process_video(
    video_path=VIDEO_PATH,
    output_csv=OUTPUT_CSV,
    interval_seconds=5.0,
    video_start_time=VIDEO_START_TIME,
)

# --- Durchschnitts-Score berechnen ----------------------------------------------
if scores:
    values = [row["score"] for row in scores]
    average_score = round(sum(values) / len(values), 2)
    print("-" * 50)
    print(f"Ausgewertete Frames : {len(values)}")
    print(f"Min / Max Score     : {min(values):.2f} / {max(values):.2f}")
    print(f"⌀ Durchschnitt      : {average_score}/100")
else:
    average_score = None
    print("Keine Frames ausgewertet - Durchschnitt nicht berechenbar.")
